# Lanjutkan training Wav2Vec2-BERT beku di laptop lain

Notebook ini melanjutkan run **`runs/20261123-190801_w2v_bert_frozen_all`**, yaitu pembanding Wav2Vec2-BERT dengan backbone beku dan probe linear (CLAUDE.md bagian 10.3), di laptop lain.

Yang dikerjakan notebook ini hanya `python -m ta.train --resume <run>`, jadi config, split, seed, dan hiperparameter tidak berubah. **Jangan mengedit `configs/` atau `config.yaml` di folder run**; resume akan menolak kalau config-nya berbeda.

## Alur kerja (pemilik ↔ teman)

1. **Aturan dan kode** hanya diubah di laptop pemilik, lalu dibagikan lewat GitHub. Teman cukup `git clone` atau `git pull`; **jangan mengedit atau commit** apa pun di repo.
2. **Bahan training**: download isi folder Drive **`Training Model/01_bahan_training/`**, lalu ekstrak ke satu folder lokal di SSD (misalnya `D:\bundle_ta`). Itulah `BUNDLE_DIR` di sel 1. Jangan menjalankan training dari folder Google Drive yang sedang sync.
3. **Training**: jalankan sel 1–6 berurutan.
4. **Hasil**: setelah training selesai, sel 8 membuat `<run>_hasil_<tanggal>.zip`. Upload zip itu ke **`Training Model/02_hasil_dari_teman/`**.
5. **Kalau training harus dihentikan sebelum selesai dan diserahkan kembali**: jalankan juga sel 9, lalu upload hasilnya ke **`02_hasil_dari_teman/checkpoint_serah_terima/`** supaya pemilik bisa melanjutkan dari epoch terakhir.

## Isi `01_bahan_training/` (= `BUNDLE_DIR`)

| Isi bundle | Ukuran | Wajib? | Fungsi |
|---|---|---|---|
| `pretrained/facebook-w2v-bert-2.0/` | ~2,3 GB | **wajib** | checkpoint `facebook/w2v-bert-2.0` revisi `da985ba` (sama persis dengan model utama) |
| `20261123-190801_w2v_bert_frozen_all/` | ~1 MB | **wajib** | folder run; `units/` berisi unit yang sudah selesai (CV S1 fold 1–3) |
| `interim/trim16k_35ecb48e72/` dan `interim/features/w2vbert_fbank_27f82ee0d4/` | ~1,7 GB | disarankan | cache audio terproses dan cache fbank; kalau ada, data mentah tidak perlu |
| notebook ini, `RUN_ON_OTHER_MACHINE.md`, `requirements-lock.txt`, `CLAUDE.md` | kecil | referensi | panduan dan aturan proyek (versi resminya tetap yang ada di GitHub) |

- Checkpoint unit yang setengah jalan (CV S1 fold 4) tidak ikut dibawa, jadi unit itu diulang dari epoch 1 (rugi sekitar 30–60 menit, hasil tetap sah).
- Kode: `git clone https://github.com/kanzennn/wav2vec2-bert-hijaiyah.git` (commit `6b0bb8f` atau yang lebih baru).
- Kalau Drive memecah unduhan folder menjadi beberapa zip, ekstrak semuanya ke folder yang sama. Hasilnya `BUNDLE_DIR` harus langsung berisi `pretrained/`, `interim/`, dan folder run.

## Syarat laptop
- GPU NVIDIA dengan VRAM minimal 8 GB, sebaiknya RTX seri 30/40 atau lebih baru (perlu dukungan bf16).
- Python 3.11 dan ruang disk kosong sekitar 15 GB.
- Laptop pemilik memakai torch 2.6.0+cu124, transformers 4.51.3, bitsandbytes 0.45.5, numpy 1.26.4, dan librosa 0.11.0. Samakan versinya.

## Kalau training perlu dihentikan
Tekan **Interrupt kernel** (■). Untuk melanjutkan, jalankan ulang sel **6**; training lanjut dari checkpoint epoch terakhir.

## 1. Atur path

In [ ]:
from pathlib import Path

REPO_DIR   = Path(r"D:\tugas-akhir")   # folder repo (hasil git clone / salinan); ganti sesuai laptop ini
BUNDLE_DIR = Path(r"D:\bundle_ta")     # hasil ekstrak Drive "Training Model/01_bahan_training" (lihat tabel di atas)
RUN_NAME   = "20261123-190801_w2v_bert_frozen_all"

RUN_DIR = REPO_DIR / "runs" / RUN_NAME
LOG     = REPO_DIR / "runs" / f"{RUN_NAME}.train.log"
assert (REPO_DIR / "pyproject.toml").exists(), f"REPO_DIR salah: {REPO_DIR}"
assert BUNDLE_DIR.exists(), f"BUNDLE_DIR tidak ada: {BUNDLE_DIR}"
print("repo  :", REPO_DIR)
print("bundle:", sorted(p.name for p in BUNDLE_DIR.iterdir()))

## 2. Instal dependensi (sekali saja)
Ubah `INSTALL = True` lalu jalankan. Versi paket diambil dari `requirements-lock.txt` (unduhan dari PyPI dan index PyTorch). Panduan lengkapnya ada di `RUN_ON_OTHER_MACHINE.md`.

In [ ]:
import sys, subprocess
INSTALL = False
if INSTALL:
    pip = [sys.executable, "-m", "pip", "install"]
    subprocess.run(pip + ["-r", str(REPO_DIR / "requirements-lock.txt")], check=True)
    subprocess.run(pip + ["-e", str(REPO_DIR), "--no-deps"], check=True)
print(sys.version)

## 3. Cek GPU dan versi

In [ ]:
import torch, transformers, numpy
print("torch", torch.__version__, "| CUDA", torch.version.cuda, "| transformers", transformers.__version__, "| numpy", numpy.__version__)
assert torch.cuda.is_available(), "CUDA tidak terdeteksi; training di CPU akan sangat lambat"
p = torch.cuda.get_device_properties(0)
print(f"GPU {p.name}, VRAM {p.total_memory/2**30:.1f} GB, bf16 {torch.cuda.is_bf16_supported()}")
assert torch.cuda.is_bf16_supported(), "GPU tidak mendukung bf16 (config memakai amp_dtype bf16); hentikan dan kabari pemilik"
try:
    import bitsandbytes
    print("bitsandbytes", bitsandbytes.__version__)
except Exception as e:
    raise SystemExit(f"bitsandbytes gagal di-import (dibutuhkan optimizer adamw8bit): {e}")
if p.total_memory / 2**30 < 7.5:
    print("PERINGATAN: VRAM < 8 GB. Kalau OOM, jangan ubah config sendiri; kabari pemilik (CLAUDE.md 10.3: batch efektif harus tetap 32).")

## 4. Salin model pretrained, folder run, dan cache

In [ ]:
import shutil, hashlib

def copy_dir(src, dst):
    if dst.exists():
        print("sudah ada, dilewati:", dst); return
    if not src.exists():
        print("tidak ada di bundle:", src); return
    print("menyalin", src, "->", dst)
    shutil.copytree(src, dst)

# 4a. checkpoint pretrained (wajib)
copy_dir(BUNDLE_DIR / "pretrained" / "facebook-w2v-bert-2.0", REPO_DIR / "pretrained" / "facebook-w2v-bert-2.0")
pt = REPO_DIR / "pretrained" / "facebook-w2v-bert-2.0"
rev = (pt / "REVISION").read_text().strip()
assert rev.startswith("da985ba"), f"revisi checkpoint {rev} != da985ba"
size = (pt / "model.safetensors").stat().st_size
assert size == 2322063736, f"ukuran model.safetensors {size} tidak sama dengan aslinya (salinan rusak?)"
print("pretrained OK, revisi", rev[:7])

# 4b. folder run (wajib)
copy_dir(BUNDLE_DIR / RUN_NAME, RUN_DIR)
assert (RUN_DIR / "run_args.json").exists() and (RUN_DIR / "config.yaml").exists(), "folder run tidak lengkap"
print("unit selesai:", len(list((RUN_DIR / "units").glob("*.json"))))
print("checkpoint :", [p.name for p in (RUN_DIR / "checkpoints").glob("*.pt")] if (RUN_DIR / "checkpoints").exists() else "tidak ada (unit setengah jalan diulang dari awal)")

# 4c. cache interim (disarankan)
for rel in ["trim16k_35ecb48e72", "features/w2vbert_fbank_27f82ee0d4"]:
    copy_dir(BUNDLE_DIR / "interim" / rel, REPO_DIR / "data" / "interim" / rel)

## 5. Data mentah (hanya kalau cache `interim/` tidak dibawa)
Sel ini mengekstrak zip dataset dengan aman: path diperiksa dulu, lalu zip bersarang diekstrak. Isi `Kode_Pipeline.zip` tidak diekstrak dan tidak dijalankan. Data mentah tidak boleh di-commit.

In [ ]:
import zipfile, io
cache_ok = (REPO_DIR / "data/interim/trim16k_35ecb48e72/stats.csv").exists() and \
           (REPO_DIR / "data/interim/features/w2vbert_fbank_27f82ee0d4/source_id.npy").exists()
raw = REPO_DIR / "data" / "raw"

def safe_extract(zf, dest):
    for n in zf.namelist():
        if n.startswith(("/", "\\")) or ".." in Path(n).parts:
            raise SystemExit(f"path berbahaya di zip: {n}")
    zf.extractall(dest)

if cache_ok:
    print("cache lengkap, data mentah tidak perlu")
elif (raw / "Data Benar").exists() and (raw / "Data Salah").exists():
    print("data/raw sudah ada")
else:
    zpath = BUNDLE_DIR / "Hijaiyah Letter Pronunciation Audio Dataset.zip"
    assert zpath.exists(), f"cache tidak ada dan zip dataset juga tidak ada: {zpath}"
    raw.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zpath) as z:
        for n in z.namelist():
            if n.endswith(("Data_Benar.zip", "Data_Salah.zip")):
                print("ekstrak", n)
                with zipfile.ZipFile(io.BytesIO(z.read(n))) as inner:
                    safe_extract(inner, raw)

if not cache_ok:
    for cat, expected in [("Data Benar", 10800), ("Data Salah", 10800)]:
        n = len(list((raw / cat).rglob("*.wav")))
        print(cat, n, "berkas WAV", "OK" if n == expected else f"!= {expected}, hentikan dan kabari pemilik")

## 5b. Tes cepat (opsional, disarankan sekali)
`test_frozen_backbone_trains_only_head` harus lulus.

In [ ]:
import os, subprocess, sys
env = {**os.environ, "PYTHONPATH": str(REPO_DIR / "src"), "PYTHONUTF8": "1"}
r = subprocess.run([sys.executable, "-m", "pytest", "-p", "no:cacheprovider", "-q", "tests/test_dl.py", "tests/test_resume.py"],
                   cwd=REPO_DIR, env=env)
print("exit", r.returncode)

## 6. Jalankan / lanjutkan training
Log lengkap disimpan di `runs/<RUN_NAME>.train.log`; notebook hanya menampilkan baris yang penting. Untuk berhenti: **Interrupt kernel**. Untuk lanjut: jalankan sel ini lagi.

In [ ]:
import os, subprocess, sys
NOISE = ("IS expected", "IS NOT", "masked_spec", "Some weights")
env = {**os.environ, "PYTHONPATH": str(REPO_DIR / "src"), "PYTHONUTF8": "1", "PYTHONUNBUFFERED": "1"}
cmd = [sys.executable, "-u", "-X", "utf8", "-X", "faulthandler", "-m", "ta.train", "--resume", str(RUN_DIR)]
print(" ".join(cmd))
with open(LOG, "a", encoding="utf-8") as log:
    proc = subprocess.Popen(cmd, cwd=REPO_DIR, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, encoding="utf-8", errors="replace", bufsize=1)
    try:
        for line in proc.stdout:
            log.write(line); log.flush()
            if not any(k in line for k in NOISE):
                print(line, end="")
        proc.wait()
        print("\nselesai, exit code", proc.returncode)
    except KeyboardInterrupt:
        proc.terminate()
        try:
            proc.wait(timeout=30)
        except subprocess.TimeoutExpired:
            proc.kill()
        print("\ndihentikan; jalankan sel ini lagi untuk lanjut dari checkpoint terakhir")

## 7. Cek progres (boleh dijalankan kapan saja, termasuk dari kernel lain)

In [ ]:
units = sorted(p.stem for p in (RUN_DIR / "units").glob("*.json"))
print(len(units), "unit selesai; terakhir:", units[-5:])
if LOG.exists():
    lines = [l for l in LOG.read_text(encoding="utf-8", errors="replace").splitlines()
             if not any(k in l for k in ("IS expected", "IS NOT", "masked_spec", "Some weights"))]
    print("\n".join(lines[-15:]))
print("selesai penuh:", (RUN_DIR / "results_rows.csv").exists())

## 8. Paket hasil untuk dikirim balik ke pemilik
Jalankan setelah sel 6 selesai, atau setelah training dihentikan. Hasilnya zip folder run **tanpa `checkpoints/`** (beberapa MB) ditambah log dan info mesin. **Upload zip ini ke Drive `Training Model/02_hasil_dari_teman/`.** Jangan ganti nama berkasnya; tanggal di nama berkas membedakan kiriman yang satu dengan yang lain.

In [ ]:
import zipfile
from datetime import datetime
out = REPO_DIR / f"{RUN_NAME}_hasil_{datetime.now():%Y%m%d-%H%M}.zip"
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    for p in RUN_DIR.rglob("*"):
        if p.is_file() and "checkpoints" not in p.relative_to(RUN_DIR).parts:
            z.write(p, Path(RUN_NAME) / p.relative_to(RUN_DIR))
    if LOG.exists():
        z.write(LOG, LOG.name)
print(out, f"{out.stat().st_size/2**20:.1f} MB")
print("selesai penuh:", (RUN_DIR / "results_rows.csv").exists(), "| unit selesai:", len(list((RUN_DIR / "units").glob("*.json"))))
print("-> upload ke Drive: Training Model/02_hasil_dari_teman/")

## 9. Serah terima checkpoint (hanya kalau training BELUM selesai dan harus dilanjutkan di laptop pemilik)
Hentikan training dulu (Interrupt kernel di sel 6). Sel ini mengumpulkan checkpoint unit yang sedang berjalan (`*.pt`, sekitar 2,3 GB per berkas). **Upload isi folder yang dicetak ke Drive `Training Model/02_hasil_dari_teman/checkpoint_serah_terima/`**, bersama zip dari sel 8. Kalau training sudah selesai penuh, sel ini tidak perlu dijalankan karena checkpoint sudah otomatis dihapus.

In [ ]:
import shutil
ck = RUN_DIR / "checkpoints"
pts = sorted(ck.glob("*.pt")) if ck.exists() else []
if not pts:
    print("tidak ada checkpoint (unit yang sedang jalan belum menyimpan epoch, atau training sudah selesai)")
else:
    out = REPO_DIR / "serah_terima" / RUN_NAME / "checkpoints"
    out.mkdir(parents=True, exist_ok=True)
    for p in pts:
        shutil.copy2(p, out / p.name)
        print(f"{p.name}: {p.stat().st_size/2**30:.2f} GB")
    print("-> upload semua berkas di", out, "ke Drive: Training Model/02_hasil_dari_teman/checkpoint_serah_terima/")